# Introduction to Multi-Agent Systems

A **single AI agent** runs one LLM in a loop with tools. A **Multi-Agent System (MAS)** coordinates **multiple LLM-powered agents**, each with its own role, prompt, and tools, to solve problems that are awkward for one generalist agent.

Think of the difference like a team at work:

- One person doing research, writing, editing, and publishing alone → **single agent**.
- A researcher, writer, editor, and project lead each doing their specialty → **multi-agent system**.

This notebook defines MAS in plain language, explains **agent roles** and **communication patterns**, contrasts **orchestration vs emergence**, and implements three coordination styles in working Python:

1. **Sequential** — fixed pipeline (A → B → C).
2. **Group chat** — agents take turns in an open conversation.
3. **Supervisor** — a router agent delegates to specialists until the task is done.

Everything is self-contained. No prior notebooks, frameworks, or external services are required.

In [ ]:
"""
Shared environment for the Engineering Release Announcement MAS demo.
Multiple agents collaborate to produce a release brief from internal sources.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

# --- Semantic sources agents can search ---
CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

RUNBOOKS: list[dict[str, str]] = [
    {
        "id": "deploy-200",
        "title": "Production deploy checklist",
        "text": "Run integration tests, apply DB migrations, deploy to canary 10%, monitor error rate 15 min, then full rollout.",
    },
    {
        "id": "comm-201",
        "title": "Customer communication standards",
        "text": "Lead with user benefit, list breaking changes separately, include rollback window and support channel #releases.",
    },
]

STYLE_GUIDE = {
    "tone": "professional, concise",
    "must_include": ["version number", "breaking changes section", "support channel"],
    "max_words": 200,
}

print(f"Changelog entries: {len(CHANGELOG)}")
print(f"Runbooks: {len(RUNBOOKS)}")

---

## 1. What Is a Multi-Agent System?

| Dimension | **Single agent** | **Multi-agent system (MAS)** |
|-----------|------------------|------------------------------|
| **Entities** | One LLM loop | Multiple agents with distinct roles |
| **Prompts** | One system message | Role-specific system prompts |
| **Coordination** | Internal chain-of-thought | Messages, handoffs, supervisor routing |
| **Best for** | Unified, short tasks | Decomposition, specialization, governance |

Formally, a MAS can be described as:

```
MAS = { agents[], communication_rules, shared_state, termination_conditions }
```

- **agents[]** — specialists (researcher, writer, critic, …).
- **communication_rules** — who talks to whom and in what order.
- **shared_state** — messages, artifacts, plans visible to some or all agents.
- **termination_conditions** — when the run stops (task done, max turns, budget).

MAS does not mean "many LLMs for everything." It means **deliberate role separation** when one agent's prompt would become too large, contradictory, or hard to debug.

---

## 2. Why Multiple Agents?

Teams split into multi-agent architectures when:

1. **Separation of concerns** — research vs writing vs safety review need different instructions.
2. **Prompt focus** — smaller, sharper system messages per role outperform one giant prompt.
3. **Parallelism** — independent subtasks (e.g. legal review + technical review) can run concurrently.
4. **Governance** — a critic or security agent can **veto** outputs before they ship.
5. **Maintainability** — update the researcher prompt without touching the writer.

### Our running example

**Goal:** Produce a v2.4.0 release announcement for customers.

| Agent | Job |
|-------|-----|
| **Researcher** | Pull changelog items and deploy runbook facts |
| **Writer** | Draft customer-facing announcement |
| **Critic** | Check style guide compliance (tone, required sections) |
| **Supervisor** | Decide which agent acts next until the brief is approved |

---

## 3. Agent Roles in MAS

Roles are conventions, not magic. Common patterns:

| Role | Responsibility | Release announcement example |
|------|----------------|------------------------------|
| **Planner** | Decompose the goal into steps | "Gather facts → draft → review → publish" |
| **Researcher** | Retrieve facts from tools / docs | Search changelog and runbooks |
| **Writer** | Produce drafts | Compose announcement email |
| **Executor** | Call external APIs / systems | Post to status page (simulated here) |
| **Critic** | Quality / safety gate | Reject draft missing breaking-changes section |
| **Supervisor** | Route to the right specialist | Send work to researcher first, then writer |
| **Human proxy** | Approve high-risk actions | Sign off before customer send |

In [ ]:
def keyword_search(docs: list[dict[str, str]], query: str, text_key: str = "item") -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for doc in docs:
        text = " ".join(str(v) for v in doc.values()).lower()
        score = sum(1 for term in terms if term in text)
        if score or not terms:
            scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [d for _, d in scored]


@dataclass
class AgentMessage:
    """Standard message passed between agents."""
    sender: str
    recipient: str
    content: str
    msg_type: str = "text"  # text | handoff | tool_result | finish
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def to_dict(self) -> dict[str, str]:
        return {
            "sender": self.sender,
            "recipient": self.recipient,
            "content": self.content,
            "type": self.msg_type,
        }


@dataclass
class SharedState:
    """Central state visible to all agents in the MAS."""
    goal: str
    messages: list[AgentMessage] = field(default_factory=list)
    artifacts: dict[str, Any] = field(default_factory=dict)
    turn_count: int = 0

    def add(self, msg: AgentMessage) -> None:
        self.messages.append(msg)
        self.turn_count += 1

    def log(self, sender: str, content: str, recipient: str = "all") -> None:
        self.add(AgentMessage(sender=sender, recipient=recipient, content=content))


class ResearcherAgent:
    """Gathers facts from changelog and runbooks."""
    name = "researcher"

    def run(self, state: SharedState) -> str:
        changes = [c["item"] for c in CHANGELOG]
        deploy = RUNBOOKS[0]["text"]
        comm = RUNBOOKS[1]["text"]
        facts = {
            "version": "2.4.0",
            "changes": changes,
            "deploy_notes": deploy,
            "comm_standards": comm,
        }
        state.artifacts["research"] = facts
        summary = (
            f"Found v{facts['version']} with {len(changes)} changes. "
            f"Deploy: {deploy[:60]}... Comm standards loaded."
        )
        state.log(self.name, summary)
        return summary


class WriterAgent:
    """Drafts the customer announcement from research artifacts."""
    name = "writer"

    def run(self, state: SharedState) -> str:
        facts = state.artifacts.get("research", {})
        changes = facts.get("changes", [])
        draft = (
            f"Release v{facts.get('version', '?')} is now available.\n\n"
            f"What's new: " + "; ".join(changes) + ".\n\n"
            f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
            f"Questions? Contact #releases."
        )
        state.artifacts["draft"] = draft
        state.log(self.name, f"Draft ready ({len(draft.split())} words)")
        return draft


class CriticAgent:
    """Reviews draft against the style guide; approves or rejects."""
    name = "critic"

    def run(self, state: SharedState) -> str:
        draft = state.artifacts.get("draft", "")
        issues = []
        for required in STYLE_GUIDE["must_include"]:
            if required == "version number" and "v2.4" not in draft:
                issues.append("missing version number")
            if required == "breaking changes section" and "Breaking changes" not in draft:
                issues.append("missing breaking changes section")
            if required == "support channel" and "#releases" not in draft:
                issues.append("missing support channel")
        word_count = len(draft.split())
        if word_count > STYLE_GUIDE["max_words"]:
            issues.append(f"too long ({word_count} words)")

        if issues:
            verdict = f"REJECTED: {', '.join(issues)}"
            state.artifacts["approved"] = False
        else:
            verdict = "APPROVED: draft meets style guide"
            state.artifacts["approved"] = True
        state.log(self.name, verdict)
        return verdict


print("Agents defined: researcher, writer, critic")

---

## 4. Communication Patterns

How agents talk determines control flow:

```
 (a) Sequential          (b) Group chat           (c) Supervisor

 A ──► B ──► C          A ◄──► B ◄──► C              Supervisor
                                                   ╱    │    ╲
                                              Research Writer Critic
```

| Pattern | Who decides next? | Auditability | Best when |
|---------|-------------------|--------------|-----------|
| **Sequential** | Fixed order | Highest | Pipeline is always the same |
| **Group chat** | Turn-taking or selector | Medium | Brainstorming, debate |
| **Supervisor** | Router agent / graph | High | Production workflows |
| **Handoff** | Agent passes explicit baton | High | OpenAI Agents SDK style |

In [ ]:
def run_sequential_pipeline(goal: str) -> SharedState:
    """Pattern (a): fixed order researcher → writer → critic."""
    state = SharedState(goal=goal)
    state.log("orchestrator", "Starting sequential pipeline")

    ResearcherAgent().run(state)
    WriterAgent().run(state)
    CriticAgent().run(state)

    state.log("orchestrator", f"Pipeline done. Approved={state.artifacts.get('approved')}")
    return state


seq_state = run_sequential_pipeline("Write v2.4.0 release announcement")
print(f"Sequential run: {seq_state.turn_count} messages, approved={seq_state.artifacts.get('approved')}")
for msg in seq_state.messages:
    print(f"  [{msg.sender}] {msg.content[:70]}{'...' if len(msg.content) > 70 else ''}")

In [ ]:
def run_group_chat(goal: str, agents: list[Any], max_rounds: int = 3) -> SharedState:
    """
    Pattern (b): round-robin group chat.
    Each agent sees the shared transcript and contributes in turn.
    """
    state = SharedState(goal=goal)
    state.log("moderator", f"Group chat started with {[a.name for a in agents]}")

    for round_num in range(max_rounds):
        for agent in agents:
            agent.run(state)
            if state.artifacts.get("approved"):
                state.log("moderator", "Critic approved — ending group chat")
                return state
    state.log("moderator", f"Max rounds ({max_rounds}) reached")
    return state


chat_state = run_group_chat(
    "Draft release announcement",
    [ResearcherAgent(), WriterAgent(), CriticAgent()],
    max_rounds=2,
)
print(f"Group chat: {chat_state.turn_count} messages")

---

## 5. The Supervisor Pattern

The **supervisor** (also called **orchestrator** or **manager**) is the most common production pattern. A router decides which specialist acts next based on shared state.

```
                    ┌─────────────────┐
                    │   SUPERVISOR    │
                    │  (router logic) │
                    └────────┬────────┘
           ┌──────────────────┼──────────────────┐
           ▼                  ▼                  ▼
   ┌───────────────┐  ┌───────────────┐  ┌───────────────┐
   │  Researcher   │  │    Writer     │  │    Critic     │
   │  gather facts │  │  draft text   │  │  approve/reject│
   └───────┬───────┘  └───────┬───────┘  └───────┬───────┘
           │                  │                  │
           └──────────────────┴──────────────────┘
                         observations
                              │
                              ▼
                    back to SUPERVISOR
                    until FINISH or max turns
```

In framework terms:

- **LangGraph** models this as a `StateGraph` with conditional edges from a supervisor node.
- **AutoGen** uses `SelectorGroupChat` where a manager picks the next speaker.
- **CrewAI** uses a **hierarchical process** with a manager agent delegating tasks.

Below is a framework-free supervisor you can read and debug line by line.

In [ ]:
AGENT_REGISTRY: dict[str, Any] = {
    "researcher": ResearcherAgent(),
    "writer": WriterAgent(),
    "critic": CriticAgent(),
}


def supervisor_route(state: SharedState) -> str:
    """
    Rule-based supervisor — production systems often use an LLM classification call here.
    Returns the next agent name or 'FINISH'.
    """
    if "research" not in state.artifacts:
        return "researcher"
    if "draft" not in state.artifacts:
        return "writer"
    if "approved" not in state.artifacts:
        return "critic"
    if state.artifacts.get("approved"):
        return "FINISH"
    # Rejected — send back to writer (one revision loop)
    if state.artifacts.get("revision_count", 0) < 1:
        state.artifacts["revision_count"] = state.artifacts.get("revision_count", 0) + 1
        del state.artifacts["draft"]
        state.artifacts.pop("approved", None)
        return "writer"
    return "FINISH"


@dataclass
class SupervisorMAS:
    """Pattern (c): supervisor routes specialists until FINISH."""

    max_turns: int = 10
    route_fn: Callable[[SharedState], str] = supervisor_route

    def run(self, goal: str) -> SharedState:
        state = SharedState(goal=goal)
        state.log("supervisor", f"Task: {goal}")

        for _ in range(self.max_turns):
            next_agent = self.route_fn(state)
            state.log("supervisor", f"Routing → {next_agent}")

            if next_agent == "FINISH":
                state.log("supervisor", "Task complete")
                break

            agent = AGENT_REGISTRY.get(next_agent)
            if not agent:
                state.log("supervisor", f"Unknown agent: {next_agent}")
                break

            agent.run(state)
        else:
            state.log("supervisor", f"Stopped: max_turns ({self.max_turns}) reached")

        return state


mas = SupervisorMAS(max_turns=8)
super_state = mas.run("Produce approved v2.4.0 release announcement")

print(f"\nFinal approved: {super_state.artifacts.get('approved')}")
print(f"Draft preview:\n{super_state.artifacts.get('draft', '(none)')[:300]}...")
print(f"\nSupervisor trace ({super_state.turn_count} messages):")
for msg in super_state.messages:
    print(f"  [{msg.sender:12}] {msg.content[:65]}{'...' if len(msg.content) > 65 else ''}")

---

## 6. Orchestration vs Emergence

| | **Orchestration** | **Emergence** |
|--|-------------------|---------------|
| **Who decides next?** | Graph, supervisor, explicit rules | Agents negotiate in open chat |
| **Auditability** | High — edges and routes are explicit | Lower — parse the transcript |
| **Flexibility** | Lower — changing flow means changing graph | Higher — conversation can drift |
| **Production fit** | Preferred for regulated / repeatable workflows | Better for brainstorming |

**Orchestration** example: our `SupervisorMAS` — the `supervisor_route` function is the control plane.

**Emergence** example: unconstrained group chat where agents decide what to say next without a fixed graph — useful for exploration but harder to guarantee completion.

Most production systems **blend both**: a supervisor orchestrates which agent speaks, while specialists may have short internal reasoning chains.

---

## 7. Shared State in MAS

Without shared state, agents repeat work or contradict each other.

| State type | Contents | In our demo |
|------------|----------|-------------|
| **Messages** | Full or summarized transcript | `SharedState.messages` |
| **Artifacts** | Drafts, plans, structured outputs | `research`, `draft`, `approved` |
| **Scratchpad** | Planner task list | Could extend `artifacts["plan"]` |
| **Tool results** | Raw API / search observations | Inside `artifacts["research"]` |

Frameworks persist this state with **checkpointers** (save after each step for resume and audit). Even in a tutorial, a single `SharedState` dataclass is the right mental model.

In [ ]:
def state_summary(state: SharedState) -> dict[str, Any]:
    senders = {}
    for msg in state.messages:
        senders[msg.sender] = senders.get(msg.sender, 0) + 1
    return {
        "goal": state.goal,
        "turn_count": state.turn_count,
        "messages_by_sender": senders,
        "artifact_keys": list(state.artifacts.keys()),
        "approved": state.artifacts.get("approved"),
    }


print(json.dumps(state_summary(super_state), indent=2))

---

## 8. Handoffs and Message Contracts

When agent A passes work to agent B, use an explicit **handoff** so the runtime (and observability tools) can trace responsibility:

```python
{
  "from": "researcher",
  "to": "writer",
  "type": "handoff",
  "content": "Research complete — 3 changelog items, deploy runbook loaded"
}
```

Clear message contracts prevent agents from misreading raw blobs and make debugging much easier.

In [ ]:
def create_handoff(from_agent: str, to_agent: str, payload: str) -> AgentMessage:
    return AgentMessage(
        sender=from_agent,
        recipient=to_agent,
        content=payload,
        msg_type="handoff",
    )


handoff = create_handoff(
    "researcher",
    "writer",
    "Research complete: v2.4.0 changelog + comm standards ready for drafting.",
)
print(json.dumps(handoff.to_dict(), indent=2))

---

## 9. Termination in MAS

Multi-agent runs must stop. Common termination conditions:

| Condition | Example |
|-----------|---------|
| **Task complete** | Supervisor routes to `FINISH`; critic returns `APPROVED` |
| **Max messages / turns** | `max_turns=10` in `SupervisorMAS` |
| **Keyword stop** | Transcript contains `APPROVED` or `DONE` |
| **Human stop** | Operator aborts from a dashboard |
| **Budget** | Token or dollar limit exceeded |

Always set at least **two** stop conditions: a success path and a hard cap.

In [ ]:
@dataclass
class TerminationPolicy:
    max_turns: int = 10
    max_cost_usd: float = 1.00
    stop_keywords: list[str] = field(default_factory=lambda: ["APPROVED", "FINISH"])

    def should_stop(self, state: SharedState, estimated_cost: float = 0.0) -> tuple[bool, str]:
        if state.turn_count >= self.max_turns:
            return True, "max_turns exceeded"
        if estimated_cost >= self.max_cost_usd:
            return True, "cost budget exceeded"
        if state.artifacts.get("approved"):
            return True, "critic approved"
        last = state.messages[-1].content if state.messages else ""
        if any(kw in last for kw in self.stop_keywords):
            return True, "stop keyword detected"
        return False, "continue"


policy = TerminationPolicy(max_turns=10)
stop, reason = policy.should_stop(super_state)
print(f"Should stop: {stop} — reason: {reason}")

---

## 10. How Frameworks Implement MAS (Conceptual Overview)

You do not need a framework to understand MAS — but teams often adopt one for checkpoints, UI, and integrations.

| Framework | Mental model | MAS feature |
|-----------|--------------|-------------|
| **LangGraph** | Directed graph of nodes | Each node = agent or tool step; supervisor = conditional edges |
| **AutoGen** | Agents in a chat room | `RoundRobinGroupChat`, `SelectorGroupChat` with a manager |
| **CrewAI** | Team org chart | Roles + tasks + sequential or hierarchical process |

### LangGraph shape

```
START → supervisor → (researcher | writer | critic) → supervisor → ... → END
```

### AutoGen shape

```
GroupChatManager picks next speaker from [researcher, writer, critic]
```

### CrewAI shape

```
Task 1 (researcher) → Task 2 (writer) → Task 3 (critic)  [sequential process]
```

Our `SupervisorMAS` is closest to LangGraph's supervisor pattern; `run_sequential_pipeline` matches CrewAI sequential; `run_group_chat` matches AutoGen round-robin.

---

## 11. Single Agent vs Multi-Agent — When to Split

| Favor **single agent** | Favor **multi-agent** |
|--------------------------|------------------------|
| Short, unified task | Conflicting expertise in one prompt |
| Low latency budget | Parallel independent subtasks |
| Simple operations / debugging | Clear organizational roles (research vs review) |
| Few tools, one domain | Governance — critic with veto power |

**Rule of thumb:** start with one agent and many tools. Split into a MAS when prompts become unmaintainable, roles conflict, or you need a separate approval gate.

In [ ]:
def recommend_architecture(signals: dict[str, bool]) -> str:
    score_multi = sum([
        signals.get("conflicting_roles", False),
        signals.get("needs_parallel_work", False),
        signals.get("needs_governance_gate", False),
        signals.get("prompt_too_large", False),
    ])
    if score_multi >= 2:
        return "multi-agent (supervisor recommended)"
    if signals.get("multi_step") and signals.get("needs_tools"):
        return "single-agent with tools"
    return "single LLM call or simple chain"


PROJECTS = [
    ("Release announcement crew", {"conflicting_roles": True, "needs_governance_gate": True, "multi_step": True}),
    ("Search docs and answer", {"multi_step": True, "needs_tools": True}),
    ("Translate a paragraph", {}),
]

for name, signals in PROJECTS:
    print(f"{recommend_architecture(signals):<35} ← {name}")

---

## 12. Agent Protocols — MCP and A2A (Preview)

As MAS grow, agents need standard ways to discover tools and talk to each other:

| Protocol | Purpose |
|----------|---------|
| **MCP (Model Context Protocol)** | Connect agents to tool/resource servers (files, DBs, APIs) through a standard interface |
| **A2A (Agent-to-Agent)** | Structured messaging between agents, possibly across services or vendors |

Design pattern: **local supervisor agent** + **remote MCP tool servers** + optional **A2A** for cross-team agents. Protocols do not replace your supervisor — they standardize the wires.

---

## 13. Executor Agent — Publishing the Final Brief

A complete MAS often ends with an **executor** that performs the irreversible action — sending email, opening a PR, updating a status page. Here we simulate publishing to an internal announcements channel.

In [ ]:
PUBLISHED_ANNOUNCEMENTS: list[dict[str, Any]] = []


class ExecutorAgent:
    """Publishes approved drafts — simulates status page / email send."""
    name = "executor"

    def run(self, state: SharedState) -> str:
        if not state.artifacts.get("approved"):
            msg = "BLOCKED: cannot publish — draft not approved by critic"
            state.log(self.name, msg)
            return msg

        draft = state.artifacts.get("draft", "")
        record = {
            "id": str(uuid.uuid4())[:8],
            "published_at": datetime.now(timezone.utc).isoformat(),
            "content": draft,
            "channel": "#releases",
        }
        PUBLISHED_ANNOUNCEMENTS.append(record)
        state.artifacts["published_id"] = record["id"]
        state.log(self.name, f"Published announcement {record['id']} to {record['channel']}")
        return f"Published: {record['id']}"


# Extend supervisor registry and run full pipeline including publish
AGENT_REGISTRY["executor"] = ExecutorAgent()


def supervisor_route_with_publish(state: SharedState) -> str:
    route = supervisor_route(state)
    if route == "FINISH" and state.artifacts.get("approved") and "published_id" not in state.artifacts:
        return "executor"
    if route == "FINISH" and "published_id" in state.artifacts:
        return "FINISH"
    return route


full_mas = SupervisorMAS(max_turns=12, route_fn=supervisor_route_with_publish)
final_state = full_mas.run("Publish v2.4.0 release announcement")

print(f"Published ID: {final_state.artifacts.get('published_id')}")
print(f"Announcements in channel: {len(PUBLISHED_ANNOUNCEMENTS)}")

---

## 14. Common MAS Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Too many agents | Cost, confusion, circular chatter | Merge roles; start with 2–3 specialists |
| No supervisor / termination | Infinite loops | Router + `max_turns` + budget |
| Duplicate tools per agent | Inconsistent behavior | Shared tool layer / MCP server |
| Unshared context | Repeated research, contradictions | Central `SharedState` or checkpointer |
| Critic without teeth | Low-quality outputs ship | Enforce reject → revise loop |
| Multi-agent for simple tasks | Latency and ops overhead | Single agent first |

---

## 15. Compare All Three Patterns Side by Side

Run the same goal through sequential, group chat, and supervisor to see behavioral differences.

In [ ]:
GOAL = "v2.4.0 release announcement"

results = {
    "sequential": run_sequential_pipeline(GOAL),
    "group_chat": run_group_chat(GOAL, [ResearcherAgent(), WriterAgent(), CriticAgent()], max_rounds=1),
    "supervisor": SupervisorMAS(max_turns=8).run(GOAL),
}

print(f"{'Pattern':<14} {'Turns':>6} {'Approved':>10} {'Artifacts'}")
print("-" * 55)
for pattern, state in results.items():
    print(
        f"{pattern:<14} {state.turn_count:>6} "
        f"{str(state.artifacts.get('approved')):>10} "
        f"{list(state.artifacts.keys())}"
    )

---

## 16. Optional — Live LLM Supervisor

Set `USE_LIVE_LLM = True` to ask a real model what a supervisor agent does. In production, that same call classifies which worker should act next.

In [ ]:
USE_LIVE_LLM = False

OFFLINE_ANSWER = (
    "A supervisor agent reads the shared task state and routes work to the right "
    "specialist — researcher, writer, or critic — until the goal is done or limits are hit."
)

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "In one sentence: what does a supervisor agent do in a multi-agent system?"}],
            max_tokens=60,
        )
        print(resp.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(OFFLINE_ANSWER)

---

## 17. Check Your Understanding

1. Define a MAS using the four components: agents, communication rules, shared state, termination.
2. What is the difference between **sequential**, **group chat**, and **supervisor** patterns?
3. Why does a **critic** agent improve governance compared to a single generalist?
4. What lives in `SharedState.artifacts` in our release announcement demo?
5. When does `supervisor_route` return `FINISH` vs send work back to the writer?
6. What is the difference between **orchestration** and **emergence**?
7. When should you stay with a **single agent** instead of splitting into a MAS?

---

## 18. Summary

A **Multi-Agent System** assigns **roles** to multiple LLM agents and coordinates them through **communication patterns** — sequential pipelines, group chat, or supervisor routing.

**Key takeaways:**

- MAS helps when **specialization**, **governance**, and **maintainable prompts** matter more than raw simplicity.
- The **supervisor pattern** is the default production shape: a router delegates to researchers, writers, critics, and executors until termination.
- **Shared state** (messages + artifacts) prevents duplicated work and enables audit trails.
- **Orchestration** (explicit routes) trades flexibility for auditability; **emergence** (open chat) does the opposite.
- **Termination** needs success conditions *and* hard caps (turns, budget, keywords).
- Frameworks like **LangGraph**, **AutoGen**, and **CrewAI** implement the same ideas with graphs, chat rooms, and org charts — our plain-Python `SupervisorMAS` mirrors the LangGraph supervisor model.
- Start with **one agent**; split only when roles conflict or prompts become unmaintainable.

You can now diagram any multi-agent product as: *Who are the specialists? Who routes? What is shared? What stops the run?*